# Evaluación del modelo híbrido

En este notebook se evaluará el modelo híbrido que combina factores latentes y popularidad.

El objetivo es analizar si la incorporación de una señal de popularidad permite mejorar la calidad de las recomendaciones Top-N respecto al modelo de factores latentes puro, manteniendo al mismo tiempo un componente de personalización.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

## 1. Carga y división de los datos

Se utiliza la misma estrategia de división por usuario empleada en las evaluaciones anteriores, reservando un 20 % de las valoraciones de cada usuario para prueba.

In [2]:
ratings = pd.read_csv("../data/raw/ml-latest-small/ratings.csv")

train_parts = []
test_parts = []

for user_id, valoraciones_usuario in ratings.groupby("userId"):

    train_user, test_user = train_test_split(
        valoraciones_usuario,
        test_size=0.2,
        random_state=42
    )

    train_parts.append(train_user)
    test_parts.append(test_user)

ratings_train = pd.concat(train_parts).reset_index(drop=True)
ratings_test = pd.concat(test_parts).reset_index(drop=True)

print("Train:", len(ratings_train))
print("Test:", len(ratings_test))

Train: 80419
Test: 20417


## 2. Preparación del componente de factores latentes

Los identificadores de usuarios y películas se transforman en índices internos utilizando exclusivamente el conjunto de entrenamiento.

In [3]:
usuarios = sorted(ratings_train["userId"].unique())
peliculas = sorted(ratings_train["movieId"].unique())

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(usuarios)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(peliculas)
}

n_usuarios = len(usuarios)
n_peliculas = len(peliculas)

ratings_train_modelo = ratings_train.copy()

ratings_train_modelo["user_idx"] = (
    ratings_train_modelo["userId"].map(user_to_idx)
)

ratings_train_modelo["movie_idx"] = (
    ratings_train_modelo["movieId"].map(movie_to_idx)
)

print("Usuarios:", n_usuarios)
print("Películas:", n_peliculas)

Usuarios: 610
Películas: 8965


## 3. Entrenamiento del componente de factores latentes

Se utiliza la configuración seleccionada durante la evaluación anterior: 20 factores latentes, tasa de aprendizaje de 0.01, regularización de 0.05 y 21 épocas.

In [4]:
n_factores = 20
learning_rate = 0.01
regularizacion = 0.05
n_epocas = 21

np.random.seed(42)

P = np.random.normal(
    scale=0.1,
    size=(n_usuarios, n_factores)
)

Q = np.random.normal(
    scale=0.1,
    size=(n_peliculas, n_factores)
)

bias_usuarios = np.zeros(n_usuarios)
bias_peliculas = np.zeros(n_peliculas)

media_global = ratings_train_modelo["rating"].mean()

for epoca in range(n_epocas):

    datos_epoca = ratings_train_modelo.sample(
        frac=1,
        random_state=42 + epoca
    )

    for fila in datos_epoca.itertuples():

        u = fila.user_idx
        i = fila.movie_idx
        rating_real = fila.rating

        prediccion = (
            media_global
            + bias_usuarios[u]
            + bias_peliculas[i]
            + np.dot(P[u], Q[i])
        )

        error = rating_real - prediccion

        P_u_anterior = P[u].copy()

        bias_usuarios[u] += learning_rate * (
            error - regularizacion * bias_usuarios[u]
        )

        bias_peliculas[i] += learning_rate * (
            error - regularizacion * bias_peliculas[i]
        )

        P[u] += learning_rate * (
            error * Q[i] - regularizacion * P[u]
        )

        Q[i] += learning_rate * (
            error * P_u_anterior - regularizacion * Q[i]
        )

    print(f"Época {epoca + 1}/{n_epocas}")

Época 1/21
Época 2/21
Época 3/21
Época 4/21
Época 5/21
Época 6/21
Época 7/21
Época 8/21
Época 9/21
Época 10/21
Época 11/21
Época 12/21
Época 13/21
Época 14/21
Época 15/21
Época 16/21
Época 17/21
Época 18/21
Época 19/21
Época 20/21
Época 21/21


## 4. Preparación de la señal de popularidad

La popularidad de cada película se calcula exclusivamente a partir del conjunto de entrenamiento para evitar utilizar información procedente del conjunto de prueba.

In [5]:
popularidad = (
    ratings_train
    .groupby("movieId")
    .size()
)

popularidad_normalizada = (
    (popularidad - popularidad.min())
    /
    (popularidad.max() - popularidad.min())
)

popularidad_dict = popularidad_normalizada.to_dict()

## 5. Evaluación de diferentes pesos del modelo híbrido

Para determinar el equilibrio más adecuado entre personalización y popularidad, se evalúan diferentes valores del parámetro `alpha`.

Para garantizar una comparación homogénea, todos los modelos se evalúan sobre los mismos usuarios y exactamente los mismos conjuntos de películas candidatas.

Se consideran relevantes las películas del conjunto de prueba con una valoración igual o superior a 4 y se utilizan 100 películas no observadas como candidatos negativos.

In [6]:
def evaluar_alphas(
    alphas,
    k=10,
    umbral_relevante=4.0,
    n_negativos=100,
    semilla=42
):

    rng = np.random.default_rng(semilla)

    resultados = {
        alpha: {
            "precision": [],
            "recall": []
        }
        for alpha in alphas
    }

    todas_peliculas = set(movie_to_idx.keys())

    for usuario_id in usuarios:

        # Películas relevantes del usuario en test
        relevantes = set(
            ratings_test.loc[
                (ratings_test["userId"] == usuario_id) &
                (ratings_test["rating"] >= umbral_relevante) &
                (ratings_test["movieId"].isin(movie_to_idx)),
                "movieId"
            ]
        )

        if len(relevantes) == 0:
            continue

        # Todas las películas que sabemos que ha valorado
        vistas = set(
            ratings.loc[
                ratings["userId"] == usuario_id,
                "movieId"
            ]
        )

        negativas_posibles = list(
            todas_peliculas - vistas
        )

        n_muestra = min(
            n_negativos,
            len(negativas_posibles)
        )

        negativas = set(
            rng.choice(
                negativas_posibles,
                size=n_muestra,
                replace=False
            )
        )

        candidatos = relevantes | negativas

        u = user_to_idx[usuario_id]

        # Calculamos los dos scores una sola vez
        scores_base = []

        for pelicula_id in candidatos:

            i = movie_to_idx[pelicula_id]

            prediccion_mf = (
                media_global
                + bias_usuarios[u]
                + bias_peliculas[i]
                + np.dot(P[u], Q[i])
            )

            prediccion_mf = np.clip(
                prediccion_mf,
                0.5,
                5.0
            )

            score_mf = (
                (prediccion_mf - 0.5)
                / 4.5
            )

            score_pop = popularidad_dict.get(
                pelicula_id,
                0.0
            )

            scores_base.append(
                (
                    pelicula_id,
                    score_mf,
                    score_pop
                )
            )

        # Probamos todos los alpha sobre
        # exactamente los mismos candidatos
        for alpha in alphas:

            scores_hibridos = [
                (
                    pelicula_id,
                    alpha * score_mf
                    + (1 - alpha) * score_pop
                )
                for pelicula_id, score_mf, score_pop
                in scores_base
            ]

            scores_hibridos.sort(
                key=lambda x: x[1],
                reverse=True
            )

            recomendadas = {
                pelicula_id
                for pelicula_id, _
                in scores_hibridos[:k]
            }

            aciertos = len(
                recomendadas.intersection(relevantes)
            )

            resultados[alpha]["precision"].append(
                aciertos / k
            )

            resultados[alpha]["recall"].append(
                aciertos / len(relevantes)
            )

    filas = []

    for alpha in alphas:

        filas.append({
            "alpha": alpha,
            "peso_MF": alpha,
            "peso_popularidad": 1 - alpha,
            "Precision@10": np.mean(
                resultados[alpha]["precision"]
            ),
            "Recall@10": np.mean(
                resultados[alpha]["recall"]
            ),
            "usuarios_evaluados": len(
                resultados[alpha]["precision"]
            )
        })

    return pd.DataFrame(filas)

In [7]:
alphas = [
    0.0,
    0.25,
    0.50,
    0.75,
    1.0
]

resultados_alpha = evaluar_alphas(
    alphas=alphas,
    k=10,
    umbral_relevante=4.0,
    n_negativos=100,
    semilla=42
)

resultados_alpha

,alpha,peso_MF,peso_popularidad,Precision@10,Recall@10,usuarios_evaluados
0,0.00,0.00,1.00,0.549833,0.566359,600
1,0.25,0.25,0.75,0.554500,0.571890,600
2,0.50,0.50,0.50,0.546167,0.550964,600
3,0.75,0.75,0.25,0.501167,0.486858,600
4,1.00,1.00,0.00,0.380500,0.345433,600


### Ajuste fino del peso de los componentes

Los primeros experimentos muestran que la combinación de ambas señales puede mejorar el rendimiento respecto a los modelos individuales.

El mejor resultado inicial se obtiene otorgando un peso mayor a la popularidad y utilizando los factores latentes como componente de personalización. Por ello, se realiza una segunda exploración más detallada alrededor de esta región.

In [8]:
alphas_finos = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40
]

resultados_alpha_finos = evaluar_alphas(
    alphas=alphas_finos,
    k=10,
    umbral_relevante=4.0,
    n_negativos=100,
    semilla=42
)

resultados_alpha_finos

,alpha,peso_MF,peso_popularidad,Precision@10,Recall@10,usuarios_evaluados
0,0.10,0.10,0.90,0.552500,0.570874,600
1,0.15,0.15,0.85,0.553833,0.570713,600
2,0.20,0.20,0.80,0.554333,0.573004,600
3,0.25,0.25,0.75,0.554500,0.571890,600
4,0.30,0.30,0.70,0.553333,0.567091,600
5,0.35,0.35,0.65,0.553667,0.565949,600
6,0.40,0.40,0.60,0.550667,0.560142,600


## 6. Análisis de la personalización de las recomendaciones

Además de obtener buenos resultados en Precision@10 y Recall@10, un sistema de recomendación debe ser capaz de adaptar sus resultados a las preferencias de cada usuario.

Dado que el modelo híbrido incorpora una señal importante de popularidad, se analiza el grado de coincidencia entre las recomendaciones generadas para diferentes usuarios.

In [9]:
def recomendar_hibrido_evaluacion(
    usuario_id,
    n=10,
    alpha=0.20
):

    u = user_to_idx[usuario_id]

    vistas = set(
        ratings_train.loc[
            ratings_train["userId"] == usuario_id,
            "movieId"
        ]
    )

    scores = []

    for pelicula_id in peliculas:

        if pelicula_id in vistas:
            continue

        i = movie_to_idx[pelicula_id]

        prediccion_mf = (
            media_global
            + bias_usuarios[u]
            + bias_peliculas[i]
            + np.dot(P[u], Q[i])
        )

        prediccion_mf = np.clip(
            prediccion_mf,
            0.5,
            5.0
        )

        score_mf = (
            (prediccion_mf - 0.5)
            / 4.5
        )

        score_pop = popularidad_dict.get(
            pelicula_id,
            0.0
        )

        score_hibrido = (
            alpha * score_mf
            + (1 - alpha) * score_pop
        )

        scores.append(
            (pelicula_id, score_hibrido)
        )

    scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        pelicula_id
        for pelicula_id, _
        in scores[:n]
    ]

### Coincidencia media entre usuarios

Se calcula la proporción de películas compartidas entre los Top-10 de diferentes usuarios. Una coincidencia elevada indicaría que el componente de popularidad domina las recomendaciones, mientras que una coincidencia menor reflejaría una mayor personalización.

In [10]:
usuarios_muestra = usuarios[:100]

recomendaciones_usuarios = {
    usuario_id: set(
        recomendar_hibrido_evaluacion(
            usuario_id,
            n=10,
            alpha=0.20
        )
    )
    for usuario_id in usuarios_muestra
}

coincidencias = []

for i in range(len(usuarios_muestra)):

    for j in range(i + 1, len(usuarios_muestra)):

        usuario_1 = usuarios_muestra[i]
        usuario_2 = usuarios_muestra[j]

        comunes = len(
            recomendaciones_usuarios[usuario_1]
            .intersection(
                recomendaciones_usuarios[usuario_2]
            )
        )

        coincidencias.append(
            comunes / 10
        )

coincidencia_media = np.mean(coincidencias)

print(
    f"Coincidencia media Top-10: "
    f"{coincidencia_media:.4f}"
)

print(
    f"Personalización aproximada: "
    f"{1 - coincidencia_media:.4f}"
)

Coincidencia media Top-10: 0.5396
Personalización aproximada: 0.4604



## 7. Análisis del equilibrio entre precisión y personalización

El aumento del peso de la popularidad mejora las métricas de ranking, pero puede provocar recomendaciones similares entre distintos usuarios.

Por ello, se comparan diferentes configuraciones del modelo híbrido analizando conjuntamente Precision@10, Recall@10 y el grado de personalización de las recomendaciones.

In [11]:
def calcular_personalizacion(
    alpha,
    n=10,
    n_usuarios_muestra=100
):

    usuarios_muestra = usuarios[:n_usuarios_muestra]

    recomendaciones_usuarios = {
        usuario_id: set(
            recomendar_hibrido_evaluacion(
                usuario_id,
                n=n,
                alpha=alpha
            )
        )
        for usuario_id in usuarios_muestra
    }

    coincidencias = []

    for i in range(len(usuarios_muestra)):

        for j in range(i + 1, len(usuarios_muestra)):

            usuario_1 = usuarios_muestra[i]
            usuario_2 = usuarios_muestra[j]

            comunes = len(
                recomendaciones_usuarios[usuario_1]
                .intersection(
                    recomendaciones_usuarios[usuario_2]
                )
            )

            coincidencias.append(
                comunes / n
            )

    coincidencia_media = np.mean(coincidencias)

    personalizacion = 1 - coincidencia_media

    return coincidencia_media, personalizacion

In [12]:
alphas_comparacion = [0.20, 0.50, 0.70]

resultados_personalizacion = []

for alpha in alphas_comparacion:

    coincidencia, personalizacion = calcular_personalizacion(
        alpha=alpha,
        n=10,
        n_usuarios_muestra=100
    )

    resultados_personalizacion.append({
        "alpha": alpha,
        "peso_MF": alpha,
        "peso_popularidad": 1 - alpha,
        "coincidencia_media": coincidencia,
        "personalizacion": personalizacion
    })

resultados_personalizacion_df = pd.DataFrame(
    resultados_personalizacion
)

resultados_personalizacion_df

,alpha,peso_MF,peso_popularidad,coincidencia_media,personalizacion
0,0.2,0.2,0.8,0.539636,0.460364
1,0.5,0.5,0.5,0.497818,0.502182
2,0.7,0.7,0.3,0.439939,0.560061
